# 20_model_eval — 최종 학습데이터 모델 학습 & 성능 평가 (1:1)

**이 노트북이 하는 일:** 최종 1:1 데이터로 **fingerprint만 / descriptor만 / 둘 결합** 세 가지 입력을
**같은 조건에서 학습·평가**하여, 어느 표현이 좋은지 비교한다.

**평가 방식:** 5-fold 교차검증(cross-validation) — 데이터를 5조각으로 나눠 4조각 학습·1조각 예측을 5번 돌려
모든 샘플에 대해 '학습에 안 쓰인 상태의 예측'을 얻고, 그 예측으로 지표를 계산한다.
> 참고: 이건 별도 test set을 떼어두는 방식이 아니고 하이퍼파라미터 튜닝도 하지 않는, **빠른 비교 평가**다.

**지표(논문 정의, 혼동행렬 TP/TN/FP/FN 기반):** MCC, Accuracy, Recall, Precision + ROC-AUC, PR-AUC.

**주의:** 지금 inactive의 대부분이 decoy(구조가 일부러 다름)라 점수가 매우 높게(≈0.99) 나오는데,
이는 '실력'보다 '문제가 쉬워서'다. 실전 판별력(실측 inactive 구분)은 이보다 낮음을 기억할 것.

### 셀 1 — 라이브러리 불러오기
- `RandomForestClassifier`: 학습에 쓸 분류기(트리 여러 개의 앙상블)
- `StratifiedKFold`, `cross_val_predict`: 클래스 비율을 유지한 5-fold CV와 교차검증 예측
- `matplotlib`: ROC/PR 곡선 그림 (Agg 백엔드=화면 없이 파일 저장 가능)
- `sklearn.metrics`: AUC·혼동행렬·MCC·곡선 계산 함수들

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')            # 파일 저장용(디스플레이 없어도 동작). 노트북에선 자동 표시됨
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier               # 랜덤포레스트 분류기
from sklearn.model_selection import StratifiedKFold, cross_val_predict  # 층화 5-fold CV
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             matthews_corrcoef, roc_curve, precision_recall_curve)  # 평가 지표들

### 셀 2 — 데이터 로드 & 세 가지 특징 집합 정의
최종 CSV를 읽어 정답 `y`(potency)와 입력을 만든다.
- `fp_cols`: `fp_`로 시작하는 fingerprint 열 1024개
- `desc_cols`: SMILES·potency·fingerprint를 뺀 나머지 = descriptor 열
- `FEATURES`: 'fingerprint만', 'descriptor만', '결합' 세 입력 행렬을 딕셔너리로 준비 → 아래에서 반복 평가

In [ ]:
# 최종 학습데이터 로드 + 3가지 특징 집합 정의
SRC = 'data/HSD17B13_final_training_1to1.csv'
df = pd.read_csv(SRC)
y = df['potency'].to_numpy()                             # 정답(1=active/0=inactive)

fp_cols = [c for c in df.columns if c.startswith('fp_')] # fingerprint 열 1024개
non_desc = set(['canonical_smiles', 'potency'] + fp_cols)
desc_cols = [c for c in df.columns if c not in non_desc] # 나머지 = descriptor 열
print('화합물', len(df), '| potency', dict(pd.Series(y).value_counts()))
print('fingerprint 열', len(fp_cols), '| descriptor 열', len(desc_cols))

FEATURES = {                                             # 비교할 3가지 입력
    'fingerprint':      df[fp_cols].to_numpy(),          # 구조 지문만
    'descriptor':       df[desc_cols].to_numpy(),        # 물성만
    '결합(FP+desc)':    df[fp_cols + desc_cols].to_numpy(),  # 둘 다
}

### 셀 3 — 성능 지표 함수 (혼동행렬 기반)
예측 확률(proba)을 임계값 0.5로 0/1 예측(pred)으로 바꾼 뒤, 혼동행렬의 네 값 TP/TN/FP/FN으로
논문 정의 그대로 지표를 계산한다.
- **Accuracy** = (TP+TN)/전체, **Recall** = TP/(TP+FN), **Precision** = TP/(TP+FP)
- **MCC**: 네 값을 모두 반영하는 균형 지표(불균형에 강함) — sklearn 함수 사용
- **ROC-AUC / PR-AUC**: 임계값과 무관하게 확률 순위의 품질을 보는 지표

In [ ]:
# 성능 지표 함수 (논문 정의: 혼동행렬 TP/TN/FP/FN 기반)
def metrics_from(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)                    # 확률 → 0/1 예측(임계값 0.5)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()   # 혼동행렬 4요소
    acc = (tp + tn) / (tp + tn + fp + fn)                 # (2) Accuracy
    rec = tp / (tp + fn) if (tp + fn) else 0.0            # (3) Recall
    prec = tp / (tp + fp) if (tp + fp) else 0.0           # (4) Precision
    mcc = matthews_corrcoef(y_true, pred)                 # (1) MCC
    roc = roc_auc_score(y_true, proba)                    # AUC (ROC)
    pr = average_precision_score(y_true, proba)           # PR-AUC
    return dict(ROC_AUC=roc, PR_AUC=pr, MCC=mcc, Accuracy=acc,
               Recall=rec, Precision=prec, TP=tp, TN=tn, FP=fp, FN=fn)

### 셀 4 — 3종 특징 집합 비교 (5-fold CV, RandomForest)
`make_model()`이 모델을 만든다 — 여기 인자(`n_estimators=300` 등)가 **하이퍼파라미터**(사람이 정하는 설정).
성능을 높이려면 보통 이 값들을 튜닝한다.
`cross_val_predict`로 각 입력의 교차검증 예측 확률을 얻어 `metrics_from`으로 지표를 계산하고,
표(res)로 정리해 CSV로 저장한다. `proba_store`는 아래 곡선 그리기에 재사용.

In [ ]:
# 3종 특징 집합을 같은 조건(5-fold CV, RandomForest)으로 비교
def make_model():
    return RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                  random_state=42, n_jobs=-1)   # ← 하이퍼파라미터(튜닝 대상)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 클래스 비율 유지 5-fold
rows, proba_store = [], {}
for name, X in FEATURES.items():                          # 3가지 입력 각각에 대해
    proba = cross_val_predict(make_model(), X, y, cv=skf, # 교차검증 예측 확률(안 본 상태의 예측)
                              method='predict_proba', n_jobs=-1)[:, 1]
    proba_store[name] = proba                             # 곡선 그리기용 저장
    m = metrics_from(y, proba)                            # 지표 계산
    m = {'features': name, **m}
    rows.append(m)
    print('[%s] ROC-AUC %.3f | PR-AUC %.3f | MCC %.3f | Acc %.3f | Recall %.3f | Prec %.3f'
          % (name, m['ROC_AUC'], m['PR_AUC'], m['MCC'], m['Accuracy'], m['Recall'], m['Precision']))

res = pd.DataFrame(rows)                                  # 결과 표
OUT = 'data/HSD17B13_model_eval_1to1.csv'
res.to_csv(OUT, index=False)
print('\n결과 저장:', OUT)
print(res[['features', 'ROC_AUC', 'PR_AUC', 'MCC', 'Accuracy', 'Recall', 'Precision']]
      .round(3).to_string(index=False))

### 셀 5 — ROC / PR 곡선 그리기
- **ROC 곡선**: 임계값을 바꿔가며 TPR(=Recall) vs FPR을 그린 것. 좌상단에 붙을수록 좋음. 대각 점선=무작위(AUC 0.5).
- **PR 곡선**: Recall vs Precision. 불균형 데이터에서 특히 유용.
세 입력을 한 그림에 겹쳐 비교하고 PNG로 저장한다.

In [ ]:
# ROC / PR 커브 (3종 비교)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))            # 좌: ROC, 우: PR
for name, proba in proba_store.items():
    fpr, tpr, _ = roc_curve(y, proba)                    # ROC 좌표
    ax[0].plot(fpr, tpr, label='%s (AUC=%.3f)' % (name, roc_auc_score(y, proba)))
    pr, rc, _ = precision_recall_curve(y, proba)         # PR 좌표
    ax[1].plot(rc, pr, label='%s (PR-AUC=%.3f)' % (name, average_precision_score(y, proba)))
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
ax[0].set_title('ROC curve'); ax[0].legend(loc='lower right', fontsize=9)
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
ax[1].set_title('Precision-Recall curve'); ax[1].legend(loc='lower left', fontsize=9)
plt.tight_layout()
FIG = 'data/HSD17B13_model_eval_1to1.png'
plt.savefig(FIG, dpi=130)                                # 그림 파일 저장
plt.show()
print('그림 저장:', FIG)